<a href="https://colab.research.google.com/github/mariahamadou-cloud/predict_rh/blob/main/ai_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
#  TURNOVER PREDICTION RH — Modèle Random Forest v4
#  Dataset enrichi v3 (18 variables) | Précision : ~85.8% | Validation Croisée : ~86.7%
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# ─── 1. CHARGEMENT DES DONNÉES ────────────────────────────────────────────────
df = pd.read_csv('HR_Turnover_Enrichi_v3.csv', sep=';')

print("=" * 65)
print("  HR TURNOVER PREDICTION — Random Forest ")
print("=" * 65)
print(f"  Dataset : {df.shape[0]} candidats, {df.shape[1]} colonnes")
print(f"  Taux démission global : {df['Target_Demission_1An'].mean():.1%}")
print("=" * 65)

# ─── 2. VARIABLES RETENUES (18 FEATURES) ──────────────────────────────────────
FEATURES = [
    'Risque_Rotation_Fit',
    'Score_Fit_Culturel',
    'Taux_Rotation_Equipe_Visee',
    'Nb_Entretiens_Paralleles',
    'Delai_Acceptation_Jours',
    'Anciennete_Moyenne_Precedente_Mois',
    'Nb_Entreprises_3_Ans',
    'Age',
    'Distance_Maison_Bureau_KM',
    'Salaire_Propose',
    'Pretentions_Salariales',
    'Ecart_Salaire',
    'Score_Test_Technique',
    'Changement_Secteur',
    'Score_Engagement_Entretien',       # Variable comportementale majeure
    'Note_Verification_References',     # Variable comportementale majeure
    'Ecart_Marche_Pct',                 # Variable macro-marché
    'Clarte_Projet_Pro'                 # Variable d'alignement de carrière
]

X = df[FEATURES]
y = df['Target_Demission_1An']

# ─── 3. SÉPARATION ENTRAÎNEMENT / TEST (80 / 20) ──────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"\n  Base d'entraînement : {X_train.shape[0]} candidats")
print(f"  Base de test        : {X_test.shape[0]} candidats")

# ─── 4. MODÈLE RANDOM FOREST — PARAMÈTRES CONTRE L'OVERFITTING ────────────────
modele_rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=12,
    min_samples_split=22,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

modele_rf.fit(X_train, y_train)

# ─── 5. ÉVALUATION ET VALIDATION CROISÉE ──────────────────────────────────────
y_pred = modele_rf.predict(X_test)
y_proba = modele_rf.predict_proba(X_test)[:, 1]

precision_train = accuracy_score(y_train, modele_rf.predict(X_train))
precision_test = accuracy_score(y_test, y_pred)
scores_cv = cross_val_score(modele_rf, X, y, cv=5, scoring='accuracy')

print(f"\n{'─'*65}")
print(f"  Précision sur la base d'Entraînement : {precision_train*100:.2f}%") # Score sur les données apprises par cœur
print(f"  Précision sur la base de Test         : {precision_test*100:.2f}%")  # Score de validation réelle sur données inédites
print(f"  Écart Train - Test                    : {(precision_train-precision_test)*100:.2f} pts") # Indicateur direct d'overfitting
print(f"  Précision Cross-Val (5 blocs / plis) : {scores_cv.mean()*100:.2f}% ± {scores_cv.std()*100:.2f}%") # Moyenne de stabilité du modèle sur 5 découpages
print(f"{'─'*65}")

# Contrôle automatique de la robustesse (Calcul de la différence entre Train et Test)
ecart = (precision_train - precision_test) * 100

print(f"  Écart calculé Train/Test : {ecart:.2f} pts")
print(" " + "─"*65)

if ecart > 12:
    # Si le modèle est trop fort en entraînement par rapport au test
    print("   Écart Train/Test élevé : overfitting probable.")
elif ecart > 7:
    # Zone intermédiaire normale pour un Random Forest de cette taille
    print("  Écart modéré et acceptable pour ce volume de données.")
else:
    # Excellent signal : le modèle généralise parfaitement ses prédictions
    print("  Écart faible : excellente généralisation du modèle.")

print("\n  Rapport de classification complet :")
# Affiche la Précision, le Rappel (Recall) et le F1-Score pour chaque classe (0 et 1)
print(classification_report(y_test, y_pred, target_names=['Reste (0)', 'Démissionne (1)']))


# ─── 6. EXPORTATION DES GRAPHIQUES DE DIAGNOSTIC ──────────────────────────────
# Initialisation d'une figure Matplotlib avec 2 sous-graphiques côte à côte
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Turnover Prediction RH — Diagnostics Random Forest v4',
             fontsize=14, fontweight='bold', color='#1E3A5F')

# --- Graphique 1 : Matrice de confusion ---
cm = confusion_matrix(y_test, y_pred) # Calcul des Vrais/Faux Positifs et Vrais/Faux Négatifs
disp = ConfusionMatrixDisplay(cm, display_labels=['Reste', 'Démissionne'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues') # Dessin de la matrice sur le premier graphique
axes[0].set_title(f'Matrice de Confusion\nPrécision Test : {precision_test*100:.1f}%',
                 fontweight='bold', color='#1E3A5F')

# --- Graphique 2 : Feature Importance (Triée et colorée) ---
# Liaison des scores d'importance avec les noms des 18 variables
importances = pd.Series(modele_rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
# Attribution de couleurs (Or pour le top 4, Bleu pour le milieu, Gris pour le reste)
colors = ['#C8A951' if i < 4 else '#4A6FA5' if i < 8 else '#C9CDD3' for i in range(len(importances))]

# Génération du graphique à barres horizontales (inversé pour avoir les plus importantes en haut)
axes[1].barh(importances.index[::-1], importances.values[::-1], color=colors[::-1])
axes[1].set_title('Importance des Variables Rétrocédées', fontweight='bold', color='#1E3A5F')
axes[1].set_xlabel('Importance relative')
axes[1].axvline(0, color='#C9CDD3', linewidth=0.5) # Ligne de repère à l'origine y

# Nettoyage esthétique du graphique (suppression des bordures inutiles)
for spine in ['top', 'right']:
    axes[1].spines[spine].set_visible(False)

plt.tight_layout() # Ajustement automatique des marges
plt.savefig('resultats_modele_rf_v4.png', dpi=150, bbox_inches='tight') # Sauvegarde physique de l'image
plt.close() # Libération immédiate de la mémoire RAM
print("\n  Graphique de diagnostic sauvegardé : resultats_modele_rf_v4.png")


# ─── 7. FONCTION DE PRÉDICTION INDIVIDUELLE (INTERFACAGE API) ─────────────────
def predire_risque(candidat: dict) -> dict:
    """
    Calcule le score de risque de turnover à un an pour un profil candidat.
    Injecte à la volée les variables calculées (Feature Engineering).
    """
    # 1. Feature Engineering dynamique (Calculs faits à la volée à partir du dictionnaire reçu)
    candidat['Ecart_Salaire'] = (
        candidat['Salaire_Propose'] - candidat['Pretentions_Salariales']
    )
    # Formule mathématique combinée : Taux d'équipe × Incompatibilité culturelle
    candidat['Risque_Rotation_Fit'] = round(
        candidat['Taux_Rotation_Equipe_Visee'] *
        (10 - candidat['Score_Fit_Culturel']) / 10, 2
    )

    # 2. Alignement des formats pour l'inférence
    row = pd.DataFrame([candidat])[FEATURES] # Conversion du dictionnaire en ligne DataFrame ordonnée selon FEATURES
    proba = modele_rf.predict_proba(row)[0][1] # Extraction de la probabilité d'appartenir à la classe 1 (Démission)
    score = round(proba * 100, 1) # Conversion en pourcentage exploitable

    # 3. Qualification du seuil critique pour l'affichage de l'alerte sur l'interface (Odoo/PWA)
    if score < 40:
        niveau, couleur = 'FAIBLE', ''
    elif score < 60:
        niveau, couleur = 'MODÉRÉ', ''
    else:
        niveau, couleur = 'ÉLEVÉ', ''

    # 4. Identification des 3 inducteurs majeurs (Facteurs clés responsables du score)
    # Normalisation mathématique pour éviter que les grands nombres (ex: Salaire) écrasent les notes de 1 à 10
    vals_norm = row.values[0] / (X.max().values + 1e-9)
    contrib = vals_norm * modele_rf.feature_importances_ # Calcul du poids réel du candidat pour chaque critère
    top_idx = np.argsort(contrib)[::-1][:3] # Extraction des index des 3 plus grandes contributions
    facteurs = [FEATURES[i] for i in top_idx] # Récupération des noms des variables correspondantes

    return {
        'score_risque': score,
        'niveau': niveau,
        'couleur': couleur,
        'facteurs_cles': facteurs,
    }


# ─── 8. VÉRIFICATION SUR UN CAS TEST ──────────────────────────────────────────
if __name__ == '__main__':
    # Simulation d'un profil de candidat type reçu en format JSON ou dictionnaire
    candidat_exemple = {
        'Age': 28,
        'Distance_Maison_Bureau_KM': 35,
        'Anciennete_Moyenne_Precedente_Mois': 14,
        'Nb_Entreprises_3_Ans': 3,
        'Score_Test_Technique': 8,
        'Score_Fit_Culturel': 4,
        'Pretentions_Salariales': 45000,
        'Salaire_Propose': 42000,
        'Taux_Rotation_Equipe_Visee': 38.5,
        'Delai_Acceptation_Jours': 12,
        'Nb_Entretiens_Paralleles': 3,
        'Changement_Secteur': 1,
        'Score_Engagement_Entretien': 4.5,     # Note d'engagement faible (facteur de risque)
        'Note_Verification_References': 5.0,    # Références moyennes (facteur de risque)
        'Ecart_Marche_Pct': -8.0,
        'Clarte_Projet_Pro': 5.0,
    }

    # Appel de la fonction pour tester le comportement de la logique métier
    resultat = predire_risque(candidat_exemple)

    print(f"\n{'─'*65}")
    print(f"  DÉMO SIMULATION — Alignement Inférence Candidat")
    print(f"{'─'*65}")
    print(f"  Score de risque calculé : {resultat['score_risque']}%")
    print(f"  Niveau du Risque        : {resultat['couleur']} {resultat['niveau']}")
    print(f"  Inducteurs Majeurs      : {', '.join(resultat['facteurs_cles'])}")
    print(f"{'─'*65}\n")

  HR TURNOVER PREDICTION — Random Forest 
  Dataset : 3000 candidats, 20 colonnes
  Taux démission global : 49.4%

  Base d'entraînement : 2400 candidats
  Base de test        : 600 candidats

─────────────────────────────────────────────────────────────────
  Précision sur la base d'Entraînement : 95.33%
  Précision sur la base de Test         : 85.83%
  Écart Train - Test                    : 9.50 pts
  Précision Cross-Val (5 blocs / plis) : 86.77% ± 1.34%
─────────────────────────────────────────────────────────────────
  Écart calculé Train/Test : 9.50 pts
 ─────────────────────────────────────────────────────────────────
  Écart modéré et acceptable pour ce volume de données.

  Rapport de classification complet :
                 precision    recall  f1-score   support

      Reste (0)       0.86      0.86      0.86       307
Démissionne (1)       0.85      0.86      0.86       293

       accuracy                           0.86       600
      macro avg       0.86      0.86     

In [ ]:
import joblib  # Bibliothèque standard pour sauvegarder les modèles de Machine Learning

# ─── EXTRACTION / SAUVEGARDE DU MODÈLE POUR LA PWA ────────────────────────────
# On sauvegarde le modèle entraîné ET la liste des features pour garder l'ordre exact
pipeline_export = {
    'modele': modele_rf,
    'features': FEATURES
}

# Création physique du fichier .pkl
joblib.dump(pipeline_export, 'modele_turnover_rf.pkl')
print("\n  SUCCÈS : Fichier 'modele_turnover_rf.pkl' extrait et prêt pour la PWA !")
print("  " + "─"*65)


  SUCCÈS : Fichier 'modele_turnover_rf.pkl' extrait et prêt pour la PWA !
  ─────────────────────────────────────────────────────────────────


In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Chargement du modèle que tu as extrait
# Assure-toi que le fichier .pkl est bien dans le même dossier
composants = joblib.load('modele_turnover.pkl')
modele = composants['modele']
features = composants['features']

@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Récupération des données envoyées par Odoo
        data = request.json

        # Feature Engineering (il faut refaire les mêmes calculs que dans ton code d'entraînement)
        data['Ecart_Salaire'] = data['Salaire_Propose'] - data['Pretentions_Salariales']
        data['Risque_Rotation_Fit'] = round(data['Taux_Rotation_Equipe_Visee'] * (10 - data['Score_Fit_Culturel']) / 10, 2)

        # Conversion en DataFrame pour le modèle
        df = pd.DataFrame([data])[features]

        # Prédiction
        score = float(modele.predict_proba(df)[0][1] * 100)

        return jsonify({'score': round(score, 2)})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

if __name__ == '__main__':
    # Lance l'API sur le port 5000
    app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
